<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/08_DBSCAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

import seismicup as sup

In [ ]:
!pip -q install python-calamine

In [ ]:
!pip3 install contextily

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import os
import re
import contextily as cx #para el basemap en geopandas
import xyzservices.providers as xyz #para escoger el basemap
import seismicup as sup
import urllib.request
import matplotlib.font_manager as fm
import random

from sklearn.cluster import DBSCAN
from matplotlib.colors import LogNorm # para la escala logaritmica de los colores
from matplotlib.ticker import LogFormatter, LogLocator # escala log
from scipy import stats # regresion lineal

# Conectar al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Paths

In [ ]:
path_save_figures = ''

In [ ]:
path_datasets = '/content/SeismicUP/Datasets_'

os.listdir(path_datasets)

# Cargar catalogos

In [ ]:
lista_catalogos = ['Cat_sis_SGC_jun1993_feb2018',
                   'Cat_sis_SGC_mar2018_2024',
                   'Cat_sis_SGC_TECTO_feb2014_2024',
                   'Cat_sis_SGC_LBG_jun1993_2021']

for i in lista_catalogos:
  cantidad_archivos = len(os.listdir(os.path.join(path_datasets, i)))
  print(f'{i} = ', cantidad_archivos)

In [ ]:
df_general = pd.DataFrame(columns=['Fecha-Hora UTC',
                                   'Latitud', # grados
                                   'Longitud', # grados
                                   'Profundidad [km]',
                                   'Magnitud',
                                   'Tipo Magnitud',
                                   'Error Latitud [km]',
                                   'Error Longitud [km]',
                                   'Error Profundidad [km]',
                                   'Numero de Fases',
                                   'RMS [seg]',
                                   'Gap', # grados
                                   'Departamento',
                                   'Municipio',
                                   ])

df_general.head()

## Catalogo SGC 1

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_jun1993_feb2018')
os.listdir(path_catalogo)

df_SGC_1 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_1(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_1 = pd.concat([df_SGC_1, df_temp_1], ignore_index=True)
    df_SGC_1.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_1['Catalogo'] = 'SGC 1'
df_SGC_1['Gap'] = df_SGC_1['Gap'].astype(float)
df_SGC_1.info()

In [ ]:
df_SGC_1.head()

## Catalogo SGC 2

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_mar2018_2024')
os.listdir(path_catalogo)

df_SGC_2 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_2(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_2 = pd.concat([df_SGC_2, df_temp_1], ignore_index=True)
    df_SGC_2.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_2['Catalogo'] = 'SGC 2'
df_SGC_2['Gap'] = df_SGC_2['Gap'].astype(float)
df_SGC_2.info()

In [ ]:
df_SGC_2.head()

## Catalogo TECTO

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_TECTO_feb2014_2024')
os.listdir(path_catalogo)

df_SGC_TECTO = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx') and (i != 'excel_estaciones.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_TECTO(path_catalogo=path_catalogo,
                                                file_excel=i)

    print(df_temp_1.shape)
    df_SGC_TECTO = pd.concat([df_SGC_TECTO, df_temp_1], ignore_index=True)
    df_SGC_TECTO.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_TECTO['Catalogo'] = 'TECTO'
df_SGC_TECTO['Profundidad [km]'] = df_SGC_TECTO['Profundidad [km]'].astype(float)
df_SGC_TECTO['Gap'] = df_SGC_TECTO['Gap'].astype(float)
df_SGC_TECTO.info()

In [ ]:
df_SGC_TECTO.head()

## Catalogo LBG

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_LBG_jun1993_2021')
os.listdir(path_catalogo)

df_SGC_LBG = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_LBG(path_catalogo=path_catalogo,
                                              file_excel=i)

    print(df_temp_1.shape)
    df_SGC_LBG = pd.concat([df_SGC_LBG, df_temp_1], ignore_index=True)
    df_SGC_LBG.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_LBG['Catalogo'] = 'LBG'
df_SGC_LBG.info()

In [ ]:
df_SGC_LBG.head()

# Unir catalogos

In [ ]:
df_unido = pd.concat([df_SGC_1, df_SGC_2, df_SGC_TECTO, df_SGC_LBG], ignore_index=True)
df_unido.reset_index(drop=True, inplace=True)

df_unido['Tiempo'] = pd.to_datetime(df_unido['Fecha-Hora UTC'])

df_unido.head()

In [ ]:
df_unido['Catalogo'].unique()

In [ ]:
df_unido.info()

In [ ]:
df_unido.describe().round(2)

## Duplicados

In [ ]:
df_unido.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 22154 duplicados

In [ ]:
df_unido['Tipo Magnitud'].unique()

In [ ]:
df_unido['Tipo Magnitud 2'] = df_unido['Tipo Magnitud'].str.lower()
df_unido['Tipo Magnitud 2'].unique()

In [ ]:
df_unido.describe()

In [ ]:
df_unido.sort_values(by=['Tiempo', 'Tipo Magnitud 2'],
                     ascending=True,
                     inplace=True)

Se considerarán únicamente los eventos que cuenten con ML como tipo de magnitud.

In [ ]:
filtro_ml = df_unido['Tipo Magnitud 2'].str.contains('ml')

df_unido[filtro_ml]['Tipo Magnitud 2'].unique()

In [ ]:
df_unido.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 22154 duplicados

In [ ]:
df_filtrado = df_unido.copy()
df_filtrado.drop_duplicates(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first', inplace=True)
df_filtrado.reset_index(drop=True, inplace=True)

In [ ]:
df_filtrado.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# Ya no se tienen duplicados

In [ ]:
df_filtrado.describe().round(2)

## Geodataframe

In [ ]:
gdf_filtrado = gpd.GeoDataFrame(
    df_filtrado, geometry=gpd.points_from_xy(df_filtrado.Longitud, df_filtrado.Latitud), crs="EPSG:4326"
)

In [ ]:
gdf_filtrado = gdf_filtrado.to_crs(9377)

In [ ]:
df_filtrado['X [m]'] = gdf_filtrado.get_coordinates()['x']
df_filtrado['Y [m]'] = gdf_filtrado.get_coordinates()['y']

df_filtrado['X [km]'] = df_filtrado['X [m]'] / 1000
df_filtrado['Y [km]'] = df_filtrado['Y [m]'] / 1000

df_filtrado.head()

## Punto referencia Campo Colorado



In [ ]:
punto_xy_CampoColorado = (4919222.973, 2308553.476) # x, y en metros

# se va calcular la distancia de los eventos al centroide o punto de referencia
# del Campo Colorado solo en XY mas no en profundidad

df_filtrado['Distancia [m]'] = np.sqrt((df_filtrado['X [m]'] - punto_xy_CampoColorado[0])**2 + (df_filtrado['Y [m]'] - punto_xy_CampoColorado[1])**2)
df_filtrado['Distancia [km]'] = df_filtrado['Distancia [m]'] / 1000

df_filtrado.describe().round(2)

## Subcatalogos

In [ ]:
# # limites para el area del VMM a analizar
# limites_VMM = {'longitud': [-74.8, -73.0],
#               'latitud': [5.9, 8.0]}

# Para tomarlos todos
limites_VMM = {'longitud': [-90.0, 90],
              'latitud': [0, 20]}



In [ ]:
filtro_VMM = (df_filtrado['Longitud'] >= limites_VMM['longitud'][0]) & (df_filtrado['Longitud'] <= limites_VMM['longitud'][1]) & (df_filtrado['Latitud'] >= limites_VMM['latitud'][0]) & (df_filtrado['Latitud'] <= limites_VMM['latitud'][1])
df_VMM = df_filtrado[filtro_VMM].copy()

profunidad_corte = 60 # km Notebook 03
df_VMM = df_VMM[df_VMM['Profundidad [km]'] <= profunidad_corte]

df_VMM.reset_index(drop=True, inplace=True)
df_VMM.info()

In [ ]:
df_VMM.describe().round(2)

In [ ]:
df_VMM.columns

# DBSCAN

In [ ]:
X = df_VMM[['X [km]', 'Y [km]', 'Profundidad [km]']].values
X

## Dist 5 km / N min 100

In [ ]:
distancia_km = 5 # 5 vmm
eventos_minimos = 100 # 100/50 vmm
# 5/100 5/50 vmm
# 7/50 topocoro
clustering = DBSCAN(eps = distancia_km,
                    min_samples = eventos_minimos)

clustering.fit(X)

clustering.labels_

In [ ]:
np.unique(clustering.labels_)

In [ ]:
df_VMM['Cluster'] = clustering.labels_
df_VMM['Cluster'].value_counts()

In [ ]:
df_VMM.head()

In [ ]:
df_VMM.to_csv(os.path.join(path_save_figures, 'df_VMM_DBSCAN_5km_n100.csv'),
              sep=';',
              decimal=',',
              index=False)

In [ ]:
filter_ = df_VMM['Cluster'] != -1


plt.figure()

plt.scatter(X[:, 0][~ filter_],
            X[:, 1][~ filter_],
            c='k',
            s=0.5, alpha=0.5)

plt.scatter(X[:, 0][filter_],
            X[:, 1][filter_],
            c=clustering.labels_[filter_],
            s=2)


plt.scatter(punto_xy_CampoColorado[0]/1000,
            punto_xy_CampoColorado[1]/1000,
            c='r',
            s=10)

plt.scatter(4879889/1000,
            2293367/1000,
            c='b',
            s=10)

plt.xlim(X[:, 0].min(), X[:, 0].max())
plt.ylim(X[:, 1].min(), X[:, 1].max())

plt.axis('equal')

plt.xlabel('X [km]')
plt.ylabel('Y [km]')
plt.title('DBSCAN')

In [ ]:
df_VMM['Tiempo'] = pd.to_datetime(df_VMM['Fecha-Hora UTC'])
df_VMM['Tiempo'].dt.year

In [ ]:
df_VMM['year'] = df_VMM['Tiempo'].dt.year
df_VMM['year']

In [ ]:

for i in np.unique(clustering.labels_):
  fig, ax = plt.subplots(figsize=(10,3), nrows=1, ncols=2)

  filtro_cluster = df_VMM['Cluster'] == i
  ax[0].hist(df_VMM['year'][filtro_cluster],
             bins=np.arange(1992, 2025, 1),
             )
  ax[0].set_xlabel('Año')
  ax[0].set_ylabel('Frecuencia')
  ax[0].grid(ls='--', alpha= 0.8)

  ax[1].hist(df_VMM['Profundidad [km]'][filtro_cluster],
           bins=np.arange(0, 65, 2.5),
           )
  ax[1].set_xlabel('Profundidad [km]')
  ax[1].set_ylabel('Frecuencia')
  ax[1].grid(ls='--', alpha= 0.8)

  plt.suptitle(f'Cluster {i}')
  plt.show()
  print()

## Dist 7 km / N min 50

In [ ]:
distancia_km = 7 # 5 vmm
eventos_minimos = 50 # 100/50 vmm
# 5/100 5/50 vmm
# 7/50 topocoro
clustering = DBSCAN(eps = distancia_km,
                    min_samples = eventos_minimos)

#clustering.fit(X)
clustering.fit_predict(X)

clustering.labels_

In [ ]:
np.unique(clustering.labels_)

In [ ]:
df_VMM['Cluster'] = clustering.labels_
df_VMM['Cluster'].value_counts()

In [ ]:
df_VMM.head()

In [ ]:
df_VMM.to_csv(os.path.join(path_save_figures, 'df_VMM_DBSCAN_7km_n50.csv'),
              sep=';',
              decimal=',',
              index=False)

In [ ]:
filter_ = df_VMM['Cluster'] != -1


plt.figure()

plt.scatter(X[:, 0][~ filter_],
            X[:, 1][~ filter_],
            c='k',
            s=0.5, alpha=0.5)

plt.scatter(X[:, 0][filter_],
            X[:, 1][filter_],
            c=clustering.labels_[filter_],
            s=2)


plt.scatter(punto_xy_CampoColorado[0]/1000,
            punto_xy_CampoColorado[1]/1000,
            c='r',
            s=10)

plt.scatter(4879889/1000,
            2293367/1000,
            c='b',
            s=10)

plt.xlim(X[:, 0].min(), X[:, 0].max())
plt.ylim(X[:, 1].min(), X[:, 1].max())

plt.axis('equal')

plt.xlabel('X [km]')
plt.ylabel('Y [km]')
plt.title('DBSCAN')

In [ ]:
df_VMM['Tiempo'] = pd.to_datetime(df_VMM['Fecha-Hora UTC'])
df_VMM['Tiempo'].dt.year

In [ ]:
df_VMM['year'] = df_VMM['Tiempo'].dt.year
df_VMM['year']

In [ ]:

for i in np.unique(clustering.labels_):
  fig, ax = plt.subplots(figsize=(10,3), nrows=1, ncols=2)

  filtro_cluster = df_VMM['Cluster'] == i
  ax[0].hist(df_VMM['year'][filtro_cluster],
             bins=np.arange(1992, 2025, 1),
             )
  ax[0].set_xlabel('Año')
  ax[0].set_ylabel('Frecuencia')
  ax[0].grid(ls='--', alpha= 0.8)

  ax[1].hist(df_VMM['Profundidad [km]'][filtro_cluster],
           bins=np.arange(0, 65, 2.5),
           )
  ax[1].set_xlabel('Profundidad [km]')
  ax[1].set_ylabel('Frecuencia')
  ax[1].grid(ls='--', alpha= 0.8)

  plt.suptitle(f'Cluster {i}')
  plt.show()
  print()

# Fin